In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import os


data_dir = '/content/drive/MyDrive/Colab Notebooks/hocsau/Car_Brand_Logos'
batch_size = 32
num_epochs = 10
learning_rate = 0.001

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Chuẩn hóa ảnh (dành cho ResNet)
])


train_data = datasets.ImageFolder(os.path.join(data_dir, '/content/drive/MyDrive/Colab Notebooks/hocsau/Car_Brand_Logos/Train'), transform=transform)
test_data = datasets.ImageFolder(os.path.join(data_dir, '/content/drive/MyDrive/Colab Notebooks/hocsau/Car_Brand_Logos/Test'), transform=transform)


train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)


model = models.resnet18(pretrained=True)

num_classes = len(train_data.classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)


model = model.cuda() if torch.cuda.is_available() else model


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Hàm huấn luyện mô hình
def train_model(model, train_loader, criterion, optimizer, num_epochs):
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        for images, labels in train_loader:

            images, labels = images.cuda() if torch.cuda.is_available() else images, labels.cuda() if torch.cuda.is_available() else labels


            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()


        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}, Accuracy: {100 * correct/total:.2f}%")

def evaluate_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.cuda() if torch.cuda.is_available() else images, labels.cuda() if torch.cuda.is_available() else labels
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f"Test Accuracy: {100 * correct / total:.2f}%")


train_model(model, train_loader, criterion, optimizer, num_epochs)
evaluate_model(model, test_loader)


torch.save(model.state_dict(), '/content/drive/MyDrive/Colab Notebooks/hocsau/car_brand_logo_model.pth')

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 174MB/s]


Epoch 1/10, Loss: 0.7238, Accuracy: 76.84%
Epoch 2/10, Loss: 0.3696, Accuracy: 88.30%
Epoch 3/10, Loss: 0.3044, Accuracy: 90.29%
Epoch 4/10, Loss: 0.2133, Accuracy: 93.35%
Epoch 5/10, Loss: 0.1390, Accuracy: 95.42%
Epoch 6/10, Loss: 0.1045, Accuracy: 96.70%
Epoch 7/10, Loss: 0.1841, Accuracy: 94.67%
Epoch 8/10, Loss: 0.1571, Accuracy: 94.99%
Epoch 9/10, Loss: 0.0844, Accuracy: 97.57%
Epoch 10/10, Loss: 0.0725, Accuracy: 97.85%
Test Accuracy: 83.25%


In [3]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import datasets, models
from torch.utils.data import DataLoader

# Thiết lập thiết bị
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Định nghĩa đường dẫn
test_dir = "/content/drive/MyDrive/Colab Notebooks/hocsau/car_log_test/Test"
model_path = "/content/drive/MyDrive/Colab Notebooks/hocsau/car_brand_logo_model.pth"

# Tiền xử lý ảnh
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Load tập test
test_dataset = datasets.ImageFolder(test_dir, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Load mô hình ResNet18 đã train
model = models.resnet18(weights=None)  # Không cần pretrained
num_classes = len(test_dataset.classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)

# Load trọng số đã train
model.load_state_dict(torch.load(model_path, map_location=device))
model = model.to(device)
model.eval()

# Hàm đánh giá mô hình trên tập test
def evaluate_model(model, test_loader):
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f"Test Accuracy: {100 * correct / total:.2f}%")

# Chạy test
evaluate_model(model, test_loader)


<ipython-input-3-9a3e0a7f5e5c>:31: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=device))


Test Accuracy: 87.50%
